In [ ]:
!pip install -q transformers datasets evaluate scikit-learn


In [ ]:
# 

In [1]:
import numpy as np
import evaluate
import pandas as pd
from datasets import Dataset
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from datasets import disable_caching
import torch
disable_caching()

from utils.load_datasets import load_MR, load_Semeval2017A  # <-- upload these to Colab


/home/jimkam/slp/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATASET = 'MR'  # or 'Semeval2017A'
PRETRAINED_MODELS = [
#     'siebert/sentiment-roberta-large-english',
    'distilbert-base-uncased-finetuned-sst-2-english',
#     'textattack/bert-base-uncased-SST-2'
]

# PRETRAINED_MODELS = [# 'finiteautomata/bertweet-base-sentiment-analysis', 
#  'cardiffnlp/twitter-roberta-base-sentiment',
#  'j-hartmann/sentiment-roberta-large-english-3-classes'
#  ]

USE_SMALL_SAMPLE = False  # set True for debugging
N_EPOCHS = 40
BATCH_SIZE = 8


In [10]:
def prepare_dataset(X, y):
    return Dataset.from_dict({'text': X, 'label': y})

if DATASET == "Semeval2017A":
    X_train, y_train, X_test, y_test = load_Semeval2017A()
elif DATASET == "MR":
    X_train, y_train, X_test, y_test = load_MR()

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
n_classes = len(le.classes_)

train_set = prepare_dataset(X_train, y_train)
test_set = prepare_dataset(X_test, y_test)


In [11]:
import os
os.environ["WANDB_DISABLED"] = "true"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

metric = evaluate.load("accuracy")
results = []

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128  # or 256 depending on your data
    )

for model_name in PRETRAINED_MODELS:
    print(f"\n--- Training {model_name} ---")

    tokenizer = AutoTokenizer.from_pretrained(model_name)



    tokenized_train = train_set.map(tokenize_function, batched=True)
    tokenized_test = test_set.map(tokenize_function, batched=True)

    if USE_SMALL_SAMPLE:
        tokenized_train = tokenized_train.shuffle(seed=42).select(range(40))
        tokenized_test = tokenized_test.shuffle(seed=42).select(range(40))

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_classes)

    args = TrainingArguments(
        output_dir=f"./results/{model_name.replace('/', '_')}",
        evaluation_strategy="epoch",
        num_train_epochs=N_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        logging_dir='./logs',
        save_strategy="no"
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return metric.compute(predictions=preds, references=labels)

    n_samples = int(0.1*len(tokenized_train))
    small_train_dataset = tokenized_train.shuffle(
        seed=42).select(range(n_samples))
    # small_eval_dataset = tokenized_test.shuffle(
    #     seed=42).select(range(n_samples))

    # small_train_dataset = tokenized_train

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=small_train_dataset,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_result = trainer.evaluate()
    print(eval_result)
    results.append({
        "Model": model_name,
        "Accuracy": eval_result["eval_accuracy"]
    })



--- Training distilbert-base-uncased-finetuned-sst-2-english ---


/home/jimkam/slp/lib/python3.8/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
  4%|▍         | 126/3125 [00:11<16:51,  2.96it/s]

{'eval_loss': 0.4244271218776703, 'eval_accuracy': 0.8912386706948641, 'eval_runtime': 1.7376, 'eval_samples_per_second': 380.987, 'eval_steps_per_second': 47.767, 'epoch': 1.0}


  8%|▊         | 252/3125 [00:23<15:49,  3.03it/s]

{'eval_loss': 0.5698521733283997, 'eval_accuracy': 0.9063444108761329, 'eval_runtime': 1.6987, 'eval_samples_per_second': 389.709, 'eval_steps_per_second': 48.861, 'epoch': 2.0}


...

{'eval_loss': 1.1039183139801025, 'eval_accuracy': 0.9018126888217523, 'eval_runtime': 1.6724, 'eval_samples_per_second': 395.84, 'eval_steps_per_second': 49.629, 'epoch': 24.0}


100%|██████████| 3125/3125 [04:50<00:00, 10.75it/s]


{'eval_loss': 1.1044398546218872, 'eval_accuracy': 0.9018126888217523, 'eval_runtime': 1.6792, 'eval_samples_per_second': 394.229, 'eval_steps_per_second': 49.427, 'epoch': 25.0}
{'train_runtime': 290.6314, 'train_samples_per_second': 86.02, 'train_steps_per_second': 10.752, 'train_loss': 0.019333670192547142, 'epoch': 25.0}


100%|██████████| 83/83 [00:01<00:00, 51.07it/s]

{'eval_loss': 1.1044398546218872, 'eval_accuracy': 0.9018126888217523, 'eval_runtime': 1.6447, 'eval_samples_per_second': 402.505, 'eval_steps_per_second': 50.465, 'epoch': 25.0}


In [12]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)
results_df

,Model,Accuracy
0,distilbert-base-uncased-finetuned-sst-2-english,0.901813


In [ ]:
import torch
torch.cuda.empty_cache()